# make-iso — SC Linux

Assemble the UEFI installer ISO from its UKI and the compressed server image.

Use a working copy created by `make lab` for interactive execution. For interactive execution, edit `NOTEBOOK_ARGS` in the next cell. CI passes these arguments through `scripts/notebook.sh`.

## Locate the repository

Jupyter kernels may start in `notebooks/` or `build/notebooks/`. Use the repository root for all relative paths.

In [ ]:
from pathlib import Path as _SCPath
import os as _sc_os
_start = _SCPath.cwd().resolve()
_candidates = (_start, *_start.parents)
_project_root = next(
    (path for path in _candidates if (path / "config/build.json").is_file() and (path / "notebooks").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the SC Linux checkout")
_sc_os.chdir(_project_root)
print("Repository:", _project_root)

In [ ]:
NOTEBOOK_ARGS = globals().get('NOTEBOOK_ARGS', ["/out/installer/sc-linux-installer.efi","/out/sc-linux.raw.zst","/out/sc-linux.iso"])

## Setup

In [ ]:
"""Build a UEFI El Torito ISO with a full installer UKI and RAW payload."""

import argparse
import os
import pathlib
import shutil
import subprocess
import tempfile
import time

### main

In [ ]:
def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("uki", type=pathlib.Path)
    p.add_argument("payload", type=pathlib.Path)
    p.add_argument("output", type=pathlib.Path)
    args = p.parse_args()
    for file in (args.uki, args.payload):
        if not file.is_file() or file.stat().st_size == 0:
            raise ValueError("Missing installer input: " + str(file))
    epoch = int(os.environ["SOURCE_DATE_EPOCH"])
    with tempfile.TemporaryDirectory(prefix="scos-iso-") as tmp:
        stage = pathlib.Path(tmp)
        esp = stage / "esp.img"
        size = (
            (args.uki.stat().st_size + 64 * 1024 * 1024 + 1024 * 1024 - 1)
            // (1024 * 1024)
        ) * (1024 * 1024)
        with esp.open("wb") as f:
            f.truncate(size)
        subprocess.run(
            ["mkfs.vfat", "--invariant", "-F", "32", "-n", "SCOS_EFI", str(esp)],
            check=True,
        )
        subprocess.run(["mmd", "-i", str(esp), "::/EFI", "::/EFI/BOOT"], check=True)
        uki = stage / "BOOTX64.EFI"
        shutil.copyfile(args.uki, uki)
        os.utime(uki, (epoch, epoch))
        subprocess.run(
            ["mcopy", "-m", "-i", str(esp), str(uki), "::/EFI/BOOT/BOOTX64.EFI"],
            check=True,
        )
        uki.unlink()
        shutil.copyfile(args.payload, stage / "sc-linux.raw.zst")
        for file in stage.iterdir():
            os.utime(file, (epoch, epoch))
        stamp = time.strftime("%Y%m%d%H%M%S00", time.gmtime(epoch))
        subprocess.run(
            [
                "xorriso",
                "-as",
                "mkisofs",
                "-iso-level",
                "3",
                "-R",
                "-J",
                "-V",
                "SCOS_INSTALL",
                "--modification-date=" + stamp,
                "-e",
                "esp.img",
                "-no-emul-boot",
                "-o",
                str(args.output.resolve()),
                str(stage),
            ],
            check=True,
        )

## Execute

The final cell invokes the operation. Any exception means this step failed.

In [ ]:
if __name__ == "__main__":
    import sys
    sys.argv = ['notebooks/make-iso.ipynb', *NOTEBOOK_ARGS]
    main()